In [1]:
from sympy import Array, Symbol, tensorcontraction, tensorproduct
import black


def bending_function_sympy(Q, Bp, Bpp):
    QM = Q.reshape(4, 3).transpose()

    # Projections (3-vector each)

    yip = tensorcontraction(tensorproduct(QM, Bp), (1, 2))
    yipp = tensorcontraction(tensorproduct(QM, Bpp), (1, 2))

    def cross(a: Array, b: Array):
        """
        3d cross product
        https://reference.wolfram.com/language/ref/Cross.html
        """
        return Array([
            a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0],
        ])

    def norm3(arr: Array):
        x, y, z = arr[0], arr[1], arr[2]
        return (x**2 + y**2 + z**2) ** (0.5)

    return (norm3(cross(yip, yipp)) / (norm3(yip) ** 3)) ** 2


sympy_res = bending_function_sympy(
    Array([Symbol(f"q{i}") for i in range(1, 13)]),
    Array([Symbol(f"bp{i}") for i in range(1, 5)]),
    Array([Symbol(f"bpp{i}") for i in range(1, 5)]),
)
print(black.format_str(repr(sympy_res), mode=black.Mode(line_length=88)).strip())

(
    (
        (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
        - (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        -(bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        - (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
    )
    ** 2
) ** 1.0 / (
    (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10) ** 2
    + (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11) ** 2
    + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12) ** 2
) ** 3.0


In [2]:
import numpy as np

# create random array inputs
ng = np.random.default_rng()
# control points
q_np = ng.random(12).astype(float)
# bernstein coefficients
bp_np = ng.random(4).astype(float)
bpp_np = ng.random(4).astype(float)

In [3]:
def bending_function(Q, Bp, Bpp):
    xp = Q.__array_namespace__()
    QM = xp.reshape(Q, (4, 3)).T

    yip = xp.vecdot(QM, Bp)
    yipp = xp.vecdot(QM, Bpp)
    num = xp.linalg.vector_norm(xp.cross(yip, yipp))
    den = xp.linalg.vector_norm(yip) ** 3
    return (num / den) ** 2


bending_function(q_np, bp_np, bpp_np)

# Try sympy on random inputs to make sure they agree
(
    bending_function_sympy(
        Array(q_np.tolist()),
        Array(bp_np.tolist()),
        Array(bpp_np.tolist()),
    ),
    bending_function(q_np, bp_np, bpp_np),
)

(0.941115211528422, np.float64(0.9411152115284221))

In [4]:
import egglog
import egglog.exp.array_api as enp

Bp = enp.NDArray([enp.Value.var(f"bp{i}") for i in range(1, 5)])
Bpp = enp.NDArray([enp.Value.var(f"bpp{i}") for i in range(1, 5)])
Q = enp.NDArray([enp.Value.var(f"q{i}") for i in range(1, 13)])
res = bending_function(Q, Bp, Bpp)
res

_NDArray_1 = reshape(
    NDArray(
        RecursiveValue.vec(
            Vec(
                RecursiveValue(Value.var("q1")),
                RecursiveValue(Value.var("q2")),
                RecursiveValue(Value.var("q3")),
                RecursiveValue(Value.var("q4")),
                RecursiveValue(Value.var("q5")),
                RecursiveValue(Value.var("q6")),
                RecursiveValue(Value.var("q7")),
                RecursiveValue(Value.var("q8")),
                RecursiveValue(Value.var("q9")),
                RecursiveValue(Value.var("q10")),
                RecursiveValue(Value.var("q11")),
                RecursiveValue(Value.var("q12")),
            )
        )
    ),
    TupleInt(Vec(Int(4), Int(3))),
).T
_NDArray_2 = vecdot(
    _NDArray_1,
    NDArray(
        RecursiveValue.vec(
            Vec(
                RecursiveValue(Value.var("bp1")),
                RecursiveValue(Value.var("bp2")),
                RecursiveValue(Value.var("bp3")),
                RecursiveValue(Value.var("bp4")),
            )
        )
    ),
)
(
    vector_norm(
        cross(
            _NDArray_2,
            vecdot(
                _NDArray_1,
                NDArray(
                    RecursiveValue.vec(
                        Vec(
                            RecursiveValue(Value.var("bpp1")),
                            RecursiveValue(Value.var("bpp2")),
                            RecursiveValue(Value.var("bpp3")),
                            RecursiveValue(Value.var("bpp4")),
                        )
                    )
                ),
            ),
        )
    )
    / vector_norm(_NDArray_2) ** NDArray(RecursiveValue(Value.from_int(Int(3))))
) ** NDArray(RecursiveValue(Value.from_int(Int(2))))

In [5]:
(scalar_res := res.eval())

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [6]:
from __future__ import annotations
from dataclasses import dataclass
from functools import total_ordering


@total_ordering
@dataclass(frozen=True)
class ArithmeticCost:
    """
    Cost type where *, +, and ** are the same and then / is more than all of them.
    """

    muls: int = 0
    other: int = 0
    adds: int = 0
    subs: int = 0
    exps: int = 0

    @property
    def not_div_ops(self) -> int:
        return self.muls + self.adds + self.exps + self.subs

    def __eq__(self, other: ArithmeticCost) -> bool:
        return self.not_div_ops == other.not_div_ops and self.other == other.other

    def __gt__(self, other: ArithmeticCost) -> bool:
        if self.other != other.other:
            return self.other > other.other
        return self.not_div_ops > other.not_div_ops

    def __add__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls + other.muls,
            other=self.other + other.other,
            adds=self.adds + other.adds,
            exps=self.exps + other.exps,
            subs=self.subs + other.subs,
        )

    def __sub__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls - other.muls,
            other=self.other - other.other,
            adds=self.adds - other.adds,
            exps=self.exps - other.exps,
            subs=self.subs - other.subs,
        )


def arith_cost_model(egraph: egglog.EGraph, expr: egglog.Expr, children_costs: list[ArithmeticCost]) -> ArithmeticCost:
    # literals are free
    if egglog.get_literal_value(expr) is not None:
        return ArithmeticCost()
    match egglog.get_callable_fn(expr):
        case enp.Value.__add__:
            c = ArithmeticCost(adds=1)
        case enp.Value.__sub__:
            c = ArithmeticCost(subs=1)
        case enp.Value.__mul__:
            c = ArithmeticCost(muls=1)
        case enp.Value.__pow__:
            c = ArithmeticCost(exps=1)
        case enp.Int | enp.Value.from_int:
            c = ArithmeticCost()
        case enp.Value.var | enp.NDArray | enp.RecursiveValue.vec | enp.RecursiveValue | egglog.Vec:
            c = ArithmeticCost()
        case _:
            c = ArithmeticCost(other=1)
    return sum(children_costs, c)


egraph = egglog.EGraph()
egraph.register(scalar_res)
_, original_cost = egraph.extract(scalar_res, include_cost=True, cost_model=arith_cost_model)
print(original_cost)
scalar_res

ArithmeticCost(muls=66, other=1, adds=49, subs=3, exps=7)


_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

Now let's try distributing...

In [7]:
@egglog.ruleset
def distribute(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)
    # Push down subtraction
    yield egglog.rewrite(a - b, subsume=True).to(a + (-1) * b)


egraph.run(distribute.saturate())
distributed_res, distributed_cost = egraph.extract(scalar_res, cost_model=arith_cost_model, include_cost=True)
print(distributed_cost)
distributed_res

ArithmeticCost(muls=348, other=1, adds=106, subs=0, exps=7)


_Value_1 = Value.var("q2") * Value.var("bp1")
_Value_2 = Value.var("q3") * Value.var("bpp1")
_Value_3 = Value.var("q6") * Value.var("bpp2")
_Value_4 = Value.var("q9") * Value.var("bpp3")
_Value_5 = Value.var("q12") * Value.var("bpp4")
_Value_6 = Value.var("q5") * Value.var("bp2")
_Value_7 = Value.var("q8") * Value.var("bp3")
_Value_8 = Value.var("q11") * Value.var("bp4")
_Value_9 = Value.from_int(Int(-1))
_Value_10 = Value.var("q3") * Value.var("bp1")
_Value_11 = Value.var("q2") * Value.var("bpp1")
_Value_12 = Value.var("q5") * Value.var("bpp2")
_Value_13 = Value.var("q8") * Value.var("bpp3")
_Value_14 = Value.var("q11") * Value.var("bpp4")
_Value_15 = Value.var("q6") * Value.var("bp2")
_Value_16 = Value.var("q9") * Value.var("bp3")
_Value_17 = Value.var("q12") * Value.var("bp4")
_Value_18 = Value.var("q1") * Value.var("bpp1")
_Value_19 = Value.var("q4") * Value.var("bpp2")
_Value_20 = Value.var("q7") * Value.var("bpp3")
_Value_21 = Value.var("q10") * Value.var("bpp4")
_Value_22 = Value.var("q1") * Value.var("bp1")
_Value_23 = Value.var("q4") * Value.var("bp2")
_Value_24 = Value.var("q7") * Value.var("bp3")
_Value_25 = Value.var("q10") * Value.var("bp4")
(
    (
        _Value_1 * _Value_2
        + _Value_1 * _Value_3
        + _Value_1 * _Value_4
        + _Value_1 * _Value_5
        + (
            _Value_6 * _Value_2
            + _Value_6 * _Value_3
            + _Value_6 * _Value_4
            + _Value_6 * _Value_5
        )
        + (
            _Value_7 * _Value_2
            + _Value_7 * _Value_3
            + _Value_7 * _Value_4
            + _Value_7 * _Value_5
        )
        + (
            _Value_8 * _Value_2
            + _Value_8 * _Value_3
            + _Value_8 * _Value_4
            + _Value_8 * _Value_5
        )
        + (
            _Value_9 * (_Value_10 * _Value_11)
            + _Value_9 * (_Value_10 * _Value_12)
            + _Value_9 * (_Value_10 * _Value_13)
            + _Value_9 * (_Value_10 * _Value_14)
            + (
                _Value_9 * (_Value_15 * _Value_11)
                + _Value_9 * (_Value_15 * _Value_12)
                + _Value_9 * (_Value_15 * _Value_13)
                + _Value_9 * (_Value_15 * _Value_14)
            )
            + (
                _Value_9 * (_Value_16 * _Value_11)
                + _Value_9 * (_Value_16 * _Value_12)
                + _Value_9 * (_Value_16 * _Value_13)
                + _Value_9 * (_Value_16 * _Value_14)
            )
            + (
                _Value_9 * (_Value_17 * _Value_11)
                + _Value_9 * (_Value_17 * _Value_12)
                + _Value_9 * (_Value_17 * _Value_13)
                + _Value_9 * (_Value_17 * _Value_14)
            )
        )
    )
    ** Value.from_int(Int(2))
    + (
        _Value_10 * _Value_18
        + _Value_10 * _Value_19
        + _Value_10 * _Value_20
        + _Value_10 * _Value_21
        + (
            _Value_15 * _Value_18
            + _Value_15 * _Value_19
            + _Value_15 * _Value_20
            + _Value_15 * _Value_21
        )
        + (
            _Value_16 * _Value_18
            + _Value_16 * _Value_19
            + _Value_16 * _Value_20
            + _Value_16 * _Value_21
        )
        + (
            _Value_17 * _Value_18
            + _Value_17 * _Value_19
            + _Value_17 * _Value_20
            + _Value_17 * _Value_21
        )
        + (
            _Value_9 * (_Value_22 * _Value_2)
            + _Value_9 * (_Value_22 * _Value_3)
            + _Value_9 * (_Value_22 * _Value_4)
            + _Value_9 * (_Value_22 * _Value_5)
            + (
                _Value_9 * (_Value_23 * _Value_2)
                + _Value_9 * (_Value_23 * _Value_3)
                + _Value_9 * (_Value_23 * _Value_4)
                + _Value_9 * (_Value_23 * _Value_5)
            )
            + (
                _Value_9 * (_Value_24 * _Value_2)
                + _Value_9 * (_Value_24 * _Value_3)
                + _Value_9 * (_Value_24 * _Value_4)
                + _Value_9 

In [8]:
egraph = egglog.EGraph()
egraph.register(distributed_res)
egraph.run(enp.to_polynomial_ruleset.saturate() + enp.factor_ruleset.saturate())
egraph.extract(distributed_res)

_Value_1 = polynomial(
    MultiSet(
        MultiSet(Value.var("q6"), Value.var("bpp2")),
        MultiSet(Value.var("q9"), Value.var("bpp3")),
        MultiSet(Value.var("bpp1"), Value.var("q3")),
        MultiSet(Value.var("q12"), Value.var("bpp4")),
    )
)
_Value_2 = polynomial(
    MultiSet(
        MultiSet(Value.var("bp2"), Value.var("q6")),
        MultiSet(Value.var("q9"), Value.var("bp3")),
        MultiSet(Value.var("bp1"), Value.var("q3")),
        MultiSet(Value.var("q12"), Value.var("bp4")),
    )
)
_Value_3 = polynomial(
    MultiSet(
        MultiSet(
            Value.var("bp1"), polynomial(MultiSet(MultiSet(Value.var("q2"), _Value_1)))
        ),
        MultiSet(
            Value.var("bp2"), polynomial(MultiSet(MultiSet(Value.var("q5"), _Value_1)))
        ),
        MultiSet(
            Value.var("bp3"), polynomial(MultiSet(MultiSet(Value.var("q8"), _Value_1)))
        ),
        MultiSet(
            Value.var("q11"), polynomial(MultiSet(MultiSet(Value.var("bp4"), _Value_1)))
        ),
        MultiSet(
            Value.from_int(Int(-1)),
            polynomial(
                MultiSet(
                    MultiSet(
                        Value.var("q11"),
                        polynomial(MultiSet(MultiSet(Value.var("bpp4"), _Value_2))),
                    ),
                    MultiSet(
                        Value.var("q8"),
                        polynomial(MultiSet(MultiSet(Value.var("bpp3"), _Value_2))),
                    ),
                    MultiSet(
                        Value.var("q2"),
                        polynomial(MultiSet(MultiSet(Value.var("bpp1"), _Value_2))),
                    ),
                    MultiSet(
                        Value.var("bpp2"),
                        polynomial(MultiSet(MultiSet(Value.var("q5"), _Value_2))),
                    ),
                )
            ),
        ),
    )
)
_Value_4 = polynomial(
    MultiSet(
        MultiSet(Value.var("q1"), Value.var("bpp1")),
        MultiSet(Value.var("bpp2"), Value.var("q4")),
        MultiSet(Value.var("bpp3"), Value.var("q7")),
        MultiSet(Value.var("bpp4"), Value.var("q10")),
    )
)
_Value_5 = polynomial(
    MultiSet(
        MultiSet(
            Value.var("q3"), polynomial(MultiSet(MultiSet(Value.var("bp1"), _Value_4)))
        ),
        MultiSet(
            Value.var("q6"), polynomial(MultiSet(MultiSet(Value.var("bp2"), _Value_4)))
        ),
        MultiSet(
            Value.var("bp3"), polynomial(MultiSet(MultiSet(Value.var("q9"), _Value_4)))
        ),
        MultiSet(
            Value.var("bp4"), polynomial(MultiSet(MultiSet(Value.var("q12"), _Value_4)))
        ),
        MultiSet(
            Value.from_int(Int(-1)),
            polynomial(
                MultiSet(
                    MultiSet(
                        Value.var("bp1"),
                        polynomial(MultiSet(MultiSet(Value.var("q1"), _Value_1))),
                    ),
                    MultiSet(
                        Value.var("q4"),
                        polynomial(MultiSet(MultiSet(Value.var("bp2"), _Value_1))),
                    ),
                    MultiSet(
                        Value.var("q7"),
                        polynomial(MultiSet(MultiSet(Value.var("bp3"), _Value_1))),
                    ),
                    MultiSet(
                        Value.var("bp4"),
                        polynomial(MultiSet(MultiSet(Value.var("q10"), _Value_1))),
                    ),
                )
            ),
        ),
    )
)
_Value_6 = polynomial(
    MultiSet(
        MultiSet(Value.var("bp3"), Value.var("q7")),
        MultiSet(Value.var("q1"), Value.var("bp1")),
        MultiSet(Value.var("q10"), Value.var("bp4")),
        MultiSet(Value.var("bp2"), Value.var("q4")),
    )
)
_Value_7 = polynomial(
    MultiSet(
        MultiSet(
            Value.var("q11"),
            polynomial(MultiSet(MultiSet(Value.var("bpp4"), _Value_6))),
        ),
        MultiSet(
            Value

In [9]:
egraph = egglog.EGraph()
egraph.register(distributed_res)
egraph.run(enp.polynomial_schedule)
factored_res, factored_cost = egraph.extract(distributed_res, include_cost=True, cost_model=arith_cost_model)
print(factored_cost)
factored_res

ArithmeticCost(muls=380, other=1, adds=248, subs=0, exps=0)


_Value_1 = (
    Value.var("q6") * Value.var("bpp2")
    + Value.var("bpp1") * Value.var("q3")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_2 = (
    Value.var("bp2") * Value.var("q6")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("bp1") * Value.var("q3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_3 = (
    Value.var("bp1") * (Value.var("q2") * _Value_1)
    + Value.var("bp2") * (Value.var("q5") * _Value_1)
    + Value.var("bp3") * (Value.var("q8") * _Value_1)
    + Value.var("q11") * (Value.var("bp4") * _Value_1)
    + Value.from_int(Int(-1))
    * (
        Value.var("q11") * (Value.var("bpp4") * _Value_2)
        + Value.var("q8") * (Value.var("bpp3") * _Value_2)
        + Value.var("q2") * (Value.var("bpp1") * _Value_2)
        + Value.var("bpp2") * (Value.var("q5") * _Value_2)
    )
)
_Value_4 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("bpp2") * Value.var("q4")
    + Value.var("bpp3") * Value.var("q7")
    + Value.var("bpp4") * Value.var("q10")
)
_Value_5 = (
    Value.var("q3") * (Value.var("bp1") * _Value_4)
    + Value.var("q6") * (Value.var("bp2") * _Value_4)
    + Value.var("bp3") * (Value.var("q9") * _Value_4)
    + Value.var("bp4") * (Value.var("q12") * _Value_4)
    + Value.from_int(Int(-1))
    * (
        Value.var("bp1") * (Value.var("q1") * _Value_1)
        + Value.var("q4") * (Value.var("bp2") * _Value_1)
        + Value.var("q7") * (Value.var("bp3") * _Value_1)
        + Value.var("bp4") * (Value.var("q10") * _Value_1)
    )
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("bp2") * Value.var("q4")
    + Value.var("bp3") * Value.var("q7")
    + Value.var("q10") * Value.var("bp4")
)
_Value_7 = (
    Value.var("q11") * (Value.var("bpp4") * _Value_6)
    + Value.var("q8") * (Value.var("bpp3") * _Value_6)
    + Value.var("q2") * (Value.var("bpp1") * _Value_6)
    + Value.var("bpp2") * (Value.var("q5") * _Value_6)
    + Value.from_int(Int(-1))
    * (
        Value.var("bp1") * (Value.var("q2") * _Value_4)
        + Value.var("bp2") * (Value.var("q5") * _Value_4)
        + Value.var("bp3") * (Value.var("q8") * _Value_4)
        + Value.var("q11") * (Value.var("bp4") * _Value_4)
    )
)
_Value_8 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("bp4") * Value.var("q11")
)
_Value_9 = _Value_6 * _Value_6 + _Value_8 * _Value_8 + _Value_2 * _Value_2
(_Value_3 * _Value_3 + _Value_5 * _Value_5 + _Value_7 * _Value_7) / (
    _Value_9 * _Value_9 * _Value_9
)

In [10]:
original_cost.not_div_ops

125

In [11]:
factored_cost.not_div_ops

628

In [12]:
distributed_cost.not_div_ops

461

In [13]:
@egglog.ruleset
def factoring(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.birewrite((a + b) * c).to(a * c + b * c)
    yield egglog.rewrite(a * b).to(b * a)
    yield egglog.rewrite(a + b).to(b + a)


egraph = egglog.EGraph()
egraph.register(distributed_res)
egraph.run(factoring.saturate())
factored_ac_result, factored_ac_cost = egraph.extract(distributed_res, cost_model=arith_cost_model, include_cost=True)
print(factored_ac_cost)
factored_ac_result

ArithmeticCost(muls=69, other=1, adds=52, subs=0, exps=7)


_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 + _Value_3 * _Value_4 * Value.from_int(Int(-1)))
    ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 + _Value_6 * _Value_2 * Value.from_int(Int(-1)))
    ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 + _Value_1 * _Value_5 * Value.from_int(Int(-1)))
    ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [14]:
egraph.all_function_sizes()

[(Int, 3),
 (· + ·, 806),
 (· * ·, 930),
 (· ** ·, 7),
 (· / ·, 1),
 (Value.from_int, 3),
 (Value.var, 20)]

In [15]:
factored_ac_cost.not_div_ops

128

In [16]:
scalar_res

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [17]:
original_cost.not_div_ops

125

In [18]:
(FunctionBending := res.eval())

_Value_1 = (
    Value.var("q2") * Value.var("bp1")
    + Value.var("q5") * Value.var("bp2")
    + Value.var("q8") * Value.var("bp3")
    + Value.var("q11") * Value.var("bp4")
)
_Value_2 = (
    Value.var("q3") * Value.var("bpp1")
    + Value.var("q6") * Value.var("bpp2")
    + Value.var("q9") * Value.var("bpp3")
    + Value.var("q12") * Value.var("bpp4")
)
_Value_3 = (
    Value.var("q3") * Value.var("bp1")
    + Value.var("q6") * Value.var("bp2")
    + Value.var("q9") * Value.var("bp3")
    + Value.var("q12") * Value.var("bp4")
)
_Value_4 = (
    Value.var("q2") * Value.var("bpp1")
    + Value.var("q5") * Value.var("bpp2")
    + Value.var("q8") * Value.var("bpp3")
    + Value.var("q11") * Value.var("bpp4")
)
_Value_5 = (
    Value.var("q1") * Value.var("bpp1")
    + Value.var("q4") * Value.var("bpp2")
    + Value.var("q7") * Value.var("bpp3")
    + Value.var("q10") * Value.var("bpp4")
)
_Value_6 = (
    Value.var("q1") * Value.var("bp1")
    + Value.var("q4") * Value.var("bp2")
    + Value.var("q7") * Value.var("bp3")
    + Value.var("q10") * Value.var("bp4")
)
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.from_int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.from_int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.from_int(Int(2))
) / (
    _Value_6 ** Value.from_int(Int(2))
    + _Value_1 ** Value.from_int(Int(2))
    + _Value_3 ** Value.from_int(Int(2))
) ** Value.from_int(
    Int(3)
)

In [19]:
egraph = egglog.EGraph()
GradientBending = enp.NDArray(FunctionBending).diff(Q)
egraph.register(GradientBending)
egraph.run(enp.array_api_schedule)
GradientBending = egraph.extract(GradientBending)

egraph = egglog.EGraph()
egraph.register(GradientBending)
egraph.run(enp.array_api_schedule)
GradientBending = egraph.extract(GradientBending)

egraph = egglog.EGraph()
egraph.register(GradientBending)
(GradientBending, GradientBending_cost) = egraph.extract(GradientBending, include_cost=True)

GradientBending_cost

19994

In [20]:
@egglog.ruleset
def distribute_pow(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)
    # Push down subtraction
    yield egglog.rewrite(a - b, subsume=True).to(a + (-1) * b)

    # yield egglog.rewrite(a ** 2, subsume=True).to(a * a)
    # yield egglog.rewrite(a ** 3, subsume=True).to(a * a * a)


egraph = egglog.EGraph()
egraph.register(GradientBending)
egraph.run(distribute_pow.saturate())
(GradientBending_distributed, GradientBending_distributed_cost) = egraph.extract(GradientBending, include_cost=True)
GradientBending_distributed_cost

4250786

In [21]:
import time

egraph = egglog.EGraph()
GradientBending_distributed_let = egraph.let("$GradientBending_distributed", GradientBending_distributed)
start = time.perf_counter()
sizes = [egraph.all_function_sizes()]
for ruleset in [enp.to_polynomial_ruleset, enp.factor_ruleset, enp.from_polynomial_ruleset]:
    egraph.run(ruleset.saturate())
    sizes.append(egraph.all_function_sizes())
stop = time.perf_counter()
run_time = stop - start
print(run_time)


GradientBending_horner_factored, GradientBending_horner_factored_cost = egraph.extract(
    GradientBending_distributed_let, include_cost=True
)
GradientBending_horner_factored_cost

5.7710545000154525


266294

In [22]:
sum(size for _, size in sizes[-1])

128768

In [23]:
import time

egraph = egglog.EGraph()
egraph.register(GradientBending_distributed)
collected = []
for i in range(20):
    n_nodes = sum(size for _, size in egraph.all_function_sizes())
    start = time.perf_counter()
    egraph.run(factoring)
    stop = time.perf_counter()
    run_time = stop - start
    print(i, run_time)

    _, GradientBending_factored_cost = egraph.extract(GradientBending_distributed, include_cost=True)
    collected.append((n_nodes, run_time, GradientBending_factored_cost))

0 0.04556962501374073
1 0.4930388749926351
2 0.9480047919787467
3 1.0375609160109889
4 0.4304182500054594
5 0.41164454200770706
6 0.3646169579878915
7 0.24228479102021083
8 0.2022749590105377
9 0.5626846670056693
10 0.9814370420062914
11 1.8750467500067316
12 2.0867967089870945
13 2.64218687498942
14 2.941049374989234
15 3.8202354999957606
16 4.212178707995918
17 4.535596375004388
18 4.740772999997716
19 4.9021770840045065


In [24]:
collected

[(56289, 0.04556962501374073, 3974378),
 (62148, 0.4930388749926351, 3427386),
 (81202, 0.9480047919787467, 2195536),
 (101162, 1.0375609160109889, 1316193),
 (113567, 0.4304182500054594, 852344),
 (121495, 0.41164454200770706, 496204),
 (128173, 0.3646169579878915, 338610),
 (133594, 0.24228479102021083, 187444),
 (141047, 0.2022749590105377, 137271),
 (153634, 0.5626846670056693, 93453),
 (174816, 0.9814370420062914, 66874),
 (202569, 1.8750467500067316, 43520),
 (232957, 2.0867967089870945, 32477),
 (273071, 2.64218687498942, 23691),
 (322813, 2.941049374989234, 18758),
 (379407, 3.8202354999957606, 16610),
 (438802, 4.212178707995918, 15970),
 (494663, 4.535596375004388, 15759),
 (551225, 4.740772999997716, 15614),
 (611244, 4.9021770840045065, 15614)]

In [25]:
# GradientBending_horner_factored

In [26]:
# frozen = egraph.freeze()
# root = frozen.global_bindings['$GradientBending_distributed']

In [27]:
# egglog.get_callable_args(egglog.get_callable_args(frozen.outputs[egglog.get_callable_args(frozen.outputs[root][0].output)[0]][0].output)[0])

In [28]:
# @egglog.ruleset
# def non_terminate(x: enp.Value, y: enp.Value, z: enp.Value):
#     yield egglog.rewrite((x * y) * z).to(x * (y * z))
#     print(egglog.rewrite(0 * x).to(x))
#     yield egglog.rewrite(0 * x).to(x)

# init = 0 * egglog.constant("a", enp.Value)
# print(init)
# egraph = egglog.EGraph(save_egglog_string=True)
# egraph.register(init)
# egraph.run(non_terminate * 5)
# print(egraph.as_egglog_string)
# egraph.saturate(non_terminate, max=5)a

In [29]:
# x1, x2, x3, x4, x5, x6, x7, x8, x9, x10 = (egglog.constant(f"x{i}", enp.Value) for i in range(1, 11))

# (EXPR := (

# (x1 + x2 / 2 - x3/3 + x4/4)**6 *
#        (x5 + 2*x6 - 3*x7 + 4*x8)**5 * (x9 + x10)**2
#      - (x1 + x2/4 - x3/3 + x4/2)**7 *
#        (x5 + 4*x6 - 3*x7 + 2*x8)**4 * (x9 - x10)**2
# ))

In [30]:
# @egglog.ruleset
# def remove_div_exp(x: enp.Value, y: enp.Value, i: egglog.i64):
#     # Replace div with multiplcation
#     yield egglog.rewrite(x / i, subsume=True).to(x * egglog.BigRat(1, i))
#     # Replace power with multiplcation
#     yield egglog.rewrite(x ** i, subsume=True).to(x ** (i -1) * x, i != 1)
#     yield egglog.rewrite(x ** 1, subsume=True).to(x)

# egraph = egglog.EGraph()
# egraph.register(EXPR)
# egraph.run(remove_div_exp.saturate())
# egraph.extract(EXPR)

In [31]:
# egraph.run(distribute.saturate())
# egraph.extract(EXPR)